In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from IPython.display import display
import math

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Group Metadata Notebook:
#### We want to create recording groups to be able to do stats on our spike parameterization to figure out what metadata parameters result in spike waveform differences. We need to create groups for this comparison because because each recording has multiple variables (species, age, sex, brainorigin, somalayer, dendritic type), that varies for each recording. Thus, when we ran the stats for spike params prior to the group separation, we didn't know which metadata params were generating the differences/influencing the changes in spike waveform.

#### I initially was doing the grouping manually, but now doing automized version (below)

#### Note: I am ignoring age (all are adults?)and weight metadata params (for now)

#### Helper functions 

In [ ]:
# Functions to save a datrame to a pickle file and another to extract the data from the pickle file

def save_dataframe_to_pickle(dataframe, file_path):
    """
    Function to save a DataFrame as a pickle file.
    
    Args:
    - dataframe (pd.DataFrame): DataFrame to be saved.
    - file_path (str): Path to save the pickle file.
    """
    dataframe.to_pickle(file_path)
    print(f"Data frame saved to {file_path}")

def load_dataframe_from_pickle(file_path):
    """
    Function to extract a DataFrame from a pickle file.
    
    Args:
    - file_path (str): Path to the pickle file.
    
    Returns:
    - dataframe (pd.DataFrame): Loaded DataFrame.
    """
    dataframe = pd.read_pickle(file_path)
    return dataframe

def save_dict_to_pickle(dictionary, filepath):
    """
    Save a dictionary to a pickle file.

    Parameters:
        dictionary (dict): The dictionary to save.
        filepath (str): The path to the pickle file.
    """
    with open(filepath, 'wb') as f:
        pickle.dump(dictionary, f)
    print(f"Dictionary saved to {filepath}")


def load_dict_from_pickle(filepath):
    """
    Load a dictionary from a pickle file.

    Parameters:
        filepath (str): The path to the pickle file.

    Returns:
        dict: The loaded dictionary.
    """
    with open(filepath, 'rb') as f:
        dictionary = pickle.load(f)
    print(f"Dictionary loaded from {filepath}")
    return dictionary

### Read in pickle with filtered parameterized data
#### This pandas dataframe contains spike param data and animal/cell metadata for each spike(one spike per row). The dataframe has already been filtered for rsq fits (see other notebooks for rsq thresholds)

In [ ]:
allMonkey_df = load_dataframe_from_pickle(r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\allMonkey_df_filt.pkl")

In [ ]:
allMonkey_df

In [ ]:
test = allMonkey_df['dendriticType'].unique()

In [ ]:
test

In [ ]:
display(allMonkey_df.loc[allMonkey_df['dendriticType'].isna()])

### Create groups

In [ ]:


# Identify metadata columns (excluding 'Monkey ID')
metadata_cols = ['dendriticType', 'SomaLayerLoc', 'brainOrigin', 'Sex', 'Species']


# Function to create subcomparisons for varying metadata
def create_comparison_groups(df, metadata_cols):
    results = []

    for varying_col in metadata_cols:
        # Columns to keep fixed (all except the varying one)
        fixed_cols = [col for col in metadata_cols if col != varying_col]

        # Group data by fixed metadata columns
        grouped = df.groupby(fixed_cols)
        subcomparison_index = 1

        for group_name, group_data in grouped:
            # Get unique values for the varying column within the fixed group
            varying_groups = group_data[varying_col].unique()

            # Ensure there are at least two values for the varying column
            if len(varying_groups) > 1:
                # Create a single subcomparison that includes all varying groups
                subsets = {value: group_data[group_data[varying_col] == value] for value in varying_groups}
                
                results.append({
                    'Subcomparison': subcomparison_index,
                    'Varying Metadata': varying_col,
                    'Fixed Metadata': ', '.join([f"{col}={val}" for col, val in zip(fixed_cols, group_name)]) if isinstance(group_name, tuple) else f"{fixed_cols[0]}={group_name}",
                    'Groups': {value: len(subset) for value, subset in subsets.items()}
                })

                subcomparison_index += 1

    return pd.DataFrame(results)

# Run the function to create comparison groups
grouping_results_df = create_comparison_groups(allMonkey_df, metadata_cols)

# Display the table in chunks for better readability
from IPython.display import display

# Display the table in chunks if it is too large
if len(grouping_results_df) > 50:
    for i in range(0, len(grouping_results_df), 50):
        display(grouping_results_df.iloc[i:i+50])
else:
    display(grouping_results_df)



In [ ]:
grouping_results_df = grouping_results_df.drop([1, 8, 14, 20, 21, 24, 30, 35])

In [ ]:
display(grouping_results_df)

In [ ]:
grouping_results_df.info()

In [ ]:
save_dataframe_to_pickle(grouping_results_df, r"C:\Users\david\Documents\Voytek Research\spike_proj\primate Dataset\groupingResults.pkl")

In [ ]:
Metadata = ["brainOrigin", "Sex", "dendriticType", "SomaLayerLoc", "Species"]
for meta in Metadata:
    print(allMonkey_df[meta].unique())
    print(allMonkey_df[meta].nunique())